# Course Foundations: Generative AI, LangChain, and RAG

**Course author:** Jayneel Girnara

New to generative AI? Start here. This lesson introduces the vocabulary and mental models used throughout the course before you connect to a hosted model or build an application.

## What you will learn
- What a large language model (LLM) does and what it cannot guarantee.
- How prompts, message roles, tokens, and context relate to a model request.
- What LangChain adds: prompts, models, chains, tools, agents, and data abstractions.
- How document chunking, embeddings, vector search, and retrieval-augmented generation (RAG) fit together.
- Which lessons to follow next and what credentials they need.

## Before you start
Read the repository [README](../README.md) for Python setup and dependency installation. Run this notebook from the course environment. The examples here are local and do not call an LLM, search engine, or other external service, so no API key is needed.

Run the code cells from top to bottom. They are short demonstrations of concepts, not a complete chat application.

## 1. Generative AI and language models

**Generative AI** describes systems that create new content, such as text, images, audio, or code. A **large language model (LLM)** is a neural network trained on large collections of text to learn patterns in language. During generation, it repeatedly estimates which token is likely to come next, given the text so far, and appends a token to the response. This next-token process is why the result can be fluent without being a verified fact.

Training and inference are different: **training** adjusts a model's learned parameters using examples; **inference** is using the already-trained model to respond to a prompt. Most applications in this course perform inference through a provider; they do not train a model. A model is not the same thing as a chatbot product. An application sends a request to a model and decides how to collect input, add instructions or data, display the result, and handle errors.

**What an LLM is not:** it is not a live search engine, a reliable database, or a human expert. It may not know recent events, may misinterpret ambiguous requests, and may generate details that were never in its input. Treat the response as a proposal to evaluate, not automatic evidence.

### Terms you will see throughout the course

| Term | Plain-language meaning |
| --- | --- |
| **Model** | The learned system that generates or transforms content. |
| **Provider** | The service or runtime that makes a model available, such as OpenAI, Google, Anthropic, Groq, or a local Ollama server. |
| **Prompt** | The instructions, question, and supporting information sent to a model. |
| **Token** | A small unit of text processed by a model; it may be a word, part of a word, punctuation, or other text. Tokens are not the same as words. |
| **Context window** | The maximum amount of input and generated text a model can handle in one request. Limits depend on the model. |
| **Inference** | Running a trained model to produce an output. |
| **Hallucination** | A plausible-sounding answer that is unsupported or incorrect. Always verify important claims. |

## 2. Prompts and message roles

A chat request is usually represented as an ordered list of messages. The **system** message gives the assistant its general instructions, the **user** message contains the current request, and an **assistant** message is a previous or generated reply. Provider libraries sometimes call the assistant role `ai` and the user role `human` internally.

The following is only Python data: it does not contact a model.

In [1]:
foundation_messages = [
    {"role": "system", "content": "Answer clearly and use plain language."},
    {"role": "user", "content": "What is a vector embedding?"},
]

for foundation_message in foundation_messages:
    print(f"{foundation_message['role']}: {foundation_message['content']}")

system: Answer clearly and use plain language.
user: What is a vector embedding?


### Context, output, and generation settings

The model processes the messages you send plus any retrieved information and conversation history. All of this uses the model's context window. A very large prompt can exceed that limit, cost more, or bury the relevant details. **Temperature** is a generation setting that can make output more or less variable; it does not make an answer factual. The parameter range and behavior depend on the model/provider.

Prompt design and reusable templates are covered in [Lesson 02](02_prompt_chains.ipynb). Conversation history and why a model does not remember separate requests by itself are covered in [Lesson 03](03_basic_memory_with_langchain.ipynb).

### Providers, API keys, cost, and privacy

Hosted providers authenticate requests with API keys. Keep keys in a local `.env` file, never in notebook cells, source code, screenshots, or commits. The repository's `.gitignore` excludes `.env` files. Local Ollama can run a compatible model on your machine, although downloading and running models still uses local storage and compute.

Hosted requests may incur charges and send your prompt or document excerpts to that provider. Do not submit confidential or personal data unless you are authorized and understand the provider's policies. Check model availability, quotas, pricing, and data-retention terms before use.

## 3. What LangChain contributes

LangChain provides reusable interfaces for working with models and composing application steps. It does not train the model or guarantee that the model's answer is correct.

- A **chat model** wraps a provider or local runtime behind a common call pattern. Compare providers in [Lesson 01](01_langchain_with_openai.ipynb), [Lesson 05](05_ollama_app.ipynb), and [Lesson 06](06_groq_langchain.ipynb).
- A **prompt template** fills named values into reusable instructions and messages.
- A **chain** connects steps, such as prompt → model → output parser. LangChain Expression Language (LCEL) uses `|` to compose compatible steps.
- **Structured output** asks a model for data matching a schema; validation can catch missing or wrongly typed fields, but cannot prove the content is true. See [Lesson 04](04_structured_outputs.ipynb) and [Lesson 15](15_pydantic_data_validation.ipynb).
- A **tool** is a function an agent is allowed to call, described with a name, purpose, and input schema. Build one in [Lesson 08](08_basic_agents.ipynb); [Lesson 09](09_google_search_agent.ipynb) shows a search tool.
- An **agent** lets a model decide whether to use available tools and what to do next. Its decisions still need limits and review.
- A **document** commonly stores text (`page_content`) with metadata such as a filename or page number. Load documents in [Lesson 10](10_rag_load_document.ipynb).

### Local example: fill a prompt template

This cell uses LangChain to format messages only. It does not call a model or need an API key.

In [2]:
from langchain_core.prompts import ChatPromptTemplate

foundation_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Explain {topic} for a beginner in two sentences."),
        ("human", "Please explain {topic}."),
    ]
)
foundation_rendered_messages = foundation_prompt.format_messages(
    topic="retrieval-augmented generation"
)

[(message.type, message.content) for message in foundation_rendered_messages]

[('system',
  'Explain retrieval-augmented generation for a beginner in two sentences.'),
 ('human', 'Please explain retrieval-augmented generation.')]

### LCEL: compose small steps

LCEL chains can combine prompts, models, parsers, and ordinary transformations. Here we compose two local Python steps to demonstrate the pipeline shape without making an API request. Later lessons connect a real model to a chain.

In [3]:
from langchain_core.runnables import RunnableLambda

foundation_chain = (
    RunnableLambda(lambda text: text.strip())
    | RunnableLambda(lambda text: f"Question: {text}")
)

foundation_chain.invoke("  What does an embedding represent?  ")

'Question: What does an embedding represent?'

## 4. Documents, loaders, and chunks

A **document loader** reads a source such as a text file, PDF, CSV, web page, or encyclopedia page and returns document objects. A long document is usually split into **chunks** before indexing. Chunk size controls how much text goes into each piece; **overlap** repeats some text across neighboring pieces so a sentence or idea is less likely to be separated from its context.

Chunking is a trade-off: chunks that are too small lose context, while chunks that are too large may contain irrelevant material and exceed useful prompt limits.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

foundation_text = (
    "RAG retrieves relevant passages before answering. "
    "Chunks are smaller pieces of source documents. "
    "Overlap preserves some text between neighboring chunks."
)
foundation_splitter = RecursiveCharacterTextSplitter(
    chunk_size=70,
    chunk_overlap=15,
)
foundation_chunks = foundation_splitter.split_text(foundation_text)
foundation_chunks

['RAG retrieves relevant passages before answering. Chunks are smaller',
 'are smaller pieces of source documents. Overlap preserves some text',
 'some text between neighboring chunks.']

### A document is text plus metadata

Metadata helps track where a retrieved passage came from. A filename or page number can later be shown as a citation or used to filter results.

In [5]:
from langchain_core.documents import Document

foundation_document = Document(
    page_content="Embeddings represent text as numeric vectors.",
    metadata={"source": "example.txt", "page": 1},
)
print(foundation_document.page_content)
print(foundation_document.metadata)

Embeddings represent text as numeric vectors.
{'source': 'example.txt', 'page': 1}


## 5. Embeddings, vector stores, and retrieval

An **embedding** is a list of numbers that represents an input, often a sentence or document chunk. An **embedding model** creates that numeric representation from text. The model learns a space in which text with related meaning will often have vectors that are closer together than unrelated text. The exact vector length and behavior depend on the embedding model.

This is different from ordinary keyword search: a semantic search may find a passage about "car insurance" for a question about "coverage for my vehicle" even when the exact words differ. It is not magic understanding, however. The result depends on the embedding model, the quality of the source text, and how the passages were split. Texts that share keywords but mean different things can also be close.

A **vector store** keeps embeddings together with the original text and metadata. At search time, the question is embedded using the compatible embedding model; a **similarity search** compares that query vector with stored vectors and returns the closest matches. Keep the original text because the numbers are useful for comparison, not readable evidence. Similarity is not a truth score: search can miss the right passage, retrieve a misleading one, or return incomplete context.

**Where we study this next:** [Lesson 11](11_rag_splitters.ipynb) focuses on creating chunks; [Lesson 12](12_vector_embeddings.ipynb) focuses on embeddings, Chroma, and similarity search. [Lesson 10](10_rag_load_document.ipynb) comes first and shows how to load source material. We combine these pieces into RAG in [Lesson 13](13_rag_based_pdf_qna_bot.ipynb) and [Lesson 14](14_agentic_rag.ipynb).

## 6. Retrieval-augmented generation (RAG)

RAG gives a model relevant source material at answer time. A typical pipeline is:

1. Load source documents.
2. Split them into chunks and preserve useful metadata.
3. Embed the chunks and store them in a vector store.
4. Embed a user's question and retrieve matching chunks.
5. Put the question and retrieved context into a prompt.
6. Ask a model to answer from the context and, when useful, cite its sources.

RAG can make answers easier to ground in supplied documents, but it does not guarantee correctness, completeness, or privacy. A model may still misread, ignore, or overstate retrieved evidence.

### RAG chain or agentic RAG?

A **RAG chain** usually follows a fixed sequence: retrieve, build context, answer. **Agentic RAG** exposes retrieval as a tool and lets an agent choose when to call it. Agents are more flexible, but add complexity: tool access should be narrow, inputs should be validated, and actions or answers should be checked. Use a simple chain when a fixed workflow is enough. We build the fixed flow in [Lesson 13](13_rag_based_pdf_qna_bot.ipynb) and compare it with agentic retrieval in [Lesson 14](14_agentic_rag.ipynb).

## 7. Tools, agents, and memory

A tool gives a model a defined capability, such as adding numbers or searching a document index. The model proposes a tool call; application code executes it and returns the result. Do not give an agent more authority than the task needs.

**Memory** is a broad term for retaining conversation or application state. A chat model does not automatically remember independent requests. An application can pass prior messages again, or store state in a checkpointer or database. Conversation history is not the same as a document knowledge base.

In [6]:
from langchain.tools import tool

@tool
def foundation_add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

foundation_add.invoke({"a": 2, "b": 3})

5

## 8. Structured output and validation

A Pydantic model describes the shape and types expected for data. It can parse values and raise an error when they do not match constraints. Schema validation checks structure, not whether a model's claim is true.

In [7]:
from pydantic import BaseModel, Field

class FoundationBook(BaseModel):
    title: str = Field(min_length=1)
    page_count: int = Field(gt=0)

foundation_book = FoundationBook(title="Learning RAG", page_count=120)
foundation_book.model_dump()

{'title': 'Learning RAG', 'page_count': 120}

## 9. Follow the course in order

Start with model and prompt basics, then work toward retrieval and agents. The numbered sequence is designed to build on earlier lessons.

| Lesson | Topic |
| --- | --- |
| [01 — Chat models](01_langchain_with_openai.ipynb) | Provider integrations and message roles |
| [02 — Prompt chains](02_prompt_chains.ipynb) | Templates and LCEL |
| [03 — Conversation history](03_basic_memory_with_langchain.ipynb) | Stateless calls and manually managed history |
| [04 — Structured output](04_structured_outputs.ipynb) | Pydantic schemas |
| [05 — Ollama](05_ollama_app.ipynb) | Run a model through Ollama |
| [06 — Groq](06_groq_langchain.ipynb) | Call a Groq-hosted model |
| [07 — Streaming](07_streaming_response.ipynb) | Consume generated text in chunks |
| [08 — Agents and tools](08_basic_agents.ipynb) | Let an agent select Python tools |
| [09 — Search agent](09_google_search_agent.ipynb) | Add web search to an agent |
| [10 — Document loaders](10_rag_load_document.ipynb) | Read text, PDFs, CSV, and web sources |
| [11 — Text splitting](11_rag_splitters.ipynb) | Prepare documents for retrieval |
| [12 — Embeddings](12_vector_embeddings.ipynb) | Build a vector store and search it |
| [13 — PDF RAG](13_rag_based_pdf_qna_bot.ipynb) | Build a grounded question-answering chain |
| [14 — Agentic RAG](14_agentic_rag.ipynb) | Give an agent a document-retrieval tool |
| [15 — Pydantic validation](15_pydantic_data_validation.ipynb) | Validate nested data |

Some later lessons call external model or search APIs. Add only the needed keys to `.env`, and expect usage limits or charges. Ollama must be running for its lesson. Exact setup commands and credential names are in the repository [README](../README.md).

## 10. Check your understanding

Before moving on, try to answer these in your own words:

1. Why can a generated answer sound confident and still be wrong?
2. What is the difference between a document chunk and an embedding?
3. What steps happen between a user question and an answer in a basic RAG workflow?
4. Why should a model's tool access be limited?
5. Which values belong in `.env`, and which should never be committed?

### Suggested answers

1. A language model generates likely text; confidence in wording is not proof of evidence.
2. A chunk is a piece of source text. An embedding is a numeric representation used to compare meaning; the vector store should keep the source text and metadata too.
3. Retrieve relevant chunks, place them with the question in a prompt, and ask the model to answer from that context.
4. Tools can access data or perform actions, so unnecessary permissions increase risk.
5. Put private API keys in the local ignored `.env` file; never commit keys or paste them into notebook source.

## Recap

A model generates an answer from its input; LangChain helps structure and compose the application around it. Loaders and splitters prepare source documents, embeddings and vector stores support retrieval, and RAG supplies retrieved context to a model. Tools and agents add capabilities, while schemas help validate output shape. None of these remove the need to verify answers, protect data, and control cost.

**Next:** continue with [01 — Chat models and message prompts](01_langchain_with_openai.ipynb).